# BVI-Net -- export real demo samples (CPU only, no training)

Re-runs the seeded prep scripts (same splits as training) and exports a few **held-out test** inputs
for the dashboard's one-click samples: ISIC2018 dermoscopy PNGs and BraTS19 4-channel `.npy` slices
(+ FLAIR preview PNGs). Attach `tschandl/isic2018-challenge-task1-data-segmentation` and
`aryashah2k/brain-tumor-segmentation-brats-2019`. Accelerator: None. Internet: On.

In [ ]:
%cd /tmp
!rm -rf BVI-Net-ISIC && git clone -q https://github.com/Ghanasree-S/BVI-Net-ISIC.git
%cd /tmp/BVI-Net-ISIC
!pip install -q nibabel albumentations

In [ ]:
!bash data/download_isic_official_val.sh
!python data/prepare_isic_a.py --raw_dir /kaggle/input --val_raw_dir data/isic2018_val_raw --out_dir data/isic2018a --fraction 1.0
!python data/prepare_brats.py --raw_dir /kaggle/input --out_dir data/brats19a --slice_stride 2

In [ ]:
import cv2, numpy as np, shutil, zipfile
from pathlib import Path
out = Path("/kaggle/working/samples"); shutil.rmtree(out, ignore_errors=True); out.mkdir(parents=True)

# ISIC test split: pick lesions of small / medium / large size
isic = Path("data/isic2018a/test")
areas = sorted(((cv2.imread(str(m), 0) > 127).mean(), m.name) for m in (isic / "masks").glob("*.png"))
picks = [areas[int(q * (len(areas) - 1))][1] for q in (0.25, 0.5, 0.75, 0.95)]
for i, name in enumerate(picks, 1):
    shutil.copy(isic / "images" / name, out / f"skin_test_{i}.png")
    print("skin", i, name)

# BraTS test split: slices with the most tumor (WT + ET), one per patient case
brats = Path("data/brats19a/test")
scored = []
for f in (brats / "images").glob("*.npy"):
    m = f.with_suffix(".png").name
    wt = (cv2.imread(str(brats / "masks_wt" / m), 0) > 127).sum()
    et = (cv2.imread(str(brats / "masks_et" / m), 0) > 127).sum()
    scored.append((wt + 2 * et, f))
seen, k = set(), 0
for _, f in sorted(scored, key=lambda t: -t[0]):
    case = f.stem.rsplit("_", 1)[0]
    if case in seen: continue
    seen.add(case); k += 1
    arr = np.load(f)
    np.save(out / f"brain_test_{k}.npy", arr)
    cv2.imwrite(str(out / f"brain_test_{k}_flair.png"), arr[:, :, 3])
    print("brain", k, f.name, arr.shape, arr.dtype)
    if k == 3: break

with zipfile.ZipFile("/kaggle/working/demo_samples.zip", "w") as z:
    for p in sorted(out.iterdir()): z.write(p, arcname=p.name)
print(sorted(p.name for p in out.iterdir()))